<a href="https://colab.research.google.com/github/Rheaxu/story-converter-ai/blob/main/SaraStoryConverter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# ==============================================================================
# CELL 1: DATA PIPELINE ENGINE
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Installs processing dependencies (PyMuPDF, edge-tts, google-genai, PyTesseract, Pillow).
# 2. Establishes the folder structure for storybook assets.
# 3. Validates the GEMINI_API_KEY from Colab Secrets.
# 4. Validates the existence of the PDF file.
# 5. Slices the PDF matrix and collects raw page text via OCR.
# 6. Queries Gemini to dynamically identify book metadata, word banks, and story pages.
# 7. Extracts the story author from the book metadata.
# 8. Generates images for each story page without cropping.
# ==============================================================================

# @title 📚 Step 1: Configure Book Details and Extract Content { display-mode: "form" }
Folder_Safe_Name = "Tigers"  # @param {type:"string"}
PDF_Filename = "tigers.pdf"  # @param {type:"string"}
Story_Title = "Tigers"  # @param {type:"string"}

import os
import json
import subprocess
from google.colab import userdata
import math
from PIL import Image

print("⚙️ Installing processing dependencies (PyMuPDF, edge-tts, google-genai, PyTesseract, Pillow)...")
subprocess.check_call(["pip", "-q", "install", "pymupdf", "edge-tts", "google-genai", "pytesseract", "Pillow"])
subprocess.check_call(["sudo", "apt-get", "install", "-y", "tesseract-ocr"])

import fitz  # PyMuPDF
import pytesseract
from google import genai
from google.genai import types

# Establish the exact folder structures requested
project_root = Folder_Safe_Name
audio_dir = os.path.join(project_root, "storybook_audios")
images_dir = os.path.join(project_root, "storybook_images")
os.makedirs(audio_dir, exist_ok=True)
os.makedirs(images_dir, exist_ok=True)

# Verify API key placement
try:
    api_key = userdata.get('GEMINI_API_KEY')
    ai_client = genai.Client(api_key=api_key)
except Exception:
    raise ValueError("❌ Missing GEMINI_API_KEY in your Colab Secrets sidebar.")

if not os.path.exists(PDF_Filename):
    raise FileNotFoundError(f"❌ Could not find '{PDF_Filename}'. Please upload your PDF to the Colab sidebar.")

print("📖 Slicing PDF matrix and collecting raw page text via OCR...")
doc = fitz  .open(PDF_Filename)
full_story_text = ""

# This loop collects text from all physical pages for Gemini analysis using OCR.
for page_num in range(len(doc)):
    page = doc[page_num]
    actual_page_idx = page_num + 1

    # Render page to a high-resolution image (PNG)
    pix = page.get_pixmap(dpi=300) # Increased DPI for better OCR
    img_path = f"temp_page_{actual_page_idx}.png"
    pix.save(img_path)

    # Perform OCR on the image
    try:
        text = pytesseract.image_to_string(Image.open(img_path))
    except pytesseract.TesseractError as e:
        print(f"⚠️ OCR failed for page {actual_page_idx}: {e}")
        text = ""

    os.remove(img_path) # Clean up temporary image

    full_story_text += f"\n--- PDF PAGE {actual_page_idx} ---\n{text}"

# Prompt Gemini to return the title and story text as separate structural properties
prompt = f"""
You are an advanced children's literature extraction engine. Inspect the following raw book text page by page.
Identify which pages contain vocabulary review items/word banks, and which pages are active narrative content.

CRITICAL POLICY FOR BOOK METADATA:
- Identify the main book title and author(s) from the very beginning of the raw content. Place this information into a dedicated "book_metadata" object.
- The "title" should be the primary book title.
- The "author" should be a concatenated string of all authors, separated by " & ".
- Assign an "audio_file" for the full book title narration, e.g., "book_title.mp3".

CRITICAL POLICY FOR WORD BANKS:
- Do not assume there are exactly 2 word banks.
- Check the words inside every discovered word bank. If you find multiple word banks that contain identical sets of words, consolidate them and ONLY keep one copy of that word bank in your final output map.

CRITICAL POLICY FOR STORY PAGES:
- Each "story_page" object in the "story_pages" array MUST correspond to exactly one full, uncropped physical page from the PDF.
- The `page_number` for each `story_page` object MUST be a unique 1-indexed physical page number from the PDF, corresponding *directly* to the `--- PDF PAGE X ---` markers in the raw content. For example, if a `--- PDF PAGE 3 ---` marker is present, a `story_page` with `"page_number": 3` can be created.
- IGNORE any numerical pagination (e.g., '1', '2', '3') found *within* the `display_text` or `speech_script` content; these are not physical page indicators.
- The maximum valid `page_number` that can be used is {len(doc)}. You MUST NOT create any `story_page` objects with `page_number` greater than {len(doc)}.
- The total number of `story_page` objects in the "story_pages" array MUST NOT exceed the total number of physical pages in the PDF ({len(doc)}).
- The "image_file" for each story_page object MUST be named `page{{page_number}}.png` where `{{page_number}}` matches the `page_number` property of that story_page object.
- For each physical page identified as a story page, extract ALL narrative content from that physical page and break it down into logical segments (e.g., sentences or short phrases).
- Each segment should have its own "display_text", "speech_script", and "audio_file".
- Do NOT split content from a single physical PDF page into multiple "story_page" objects.
- Do NOT include "is_first_story_page", "title_text", or "audio_title_file" properties in any story_page object. All story pages are now treated as standard content pages after the initial "book_metadata".

Return ONLY a single, perfectly valid, and unadorned JSON object. Do not include any other text, comments, or markdown formatting (e.g., ```json).
Ensure all double quotes within string values are properly escaped (e.g., \"It's \"fun\").
Follow this exact format:
{{
  "book_metadata": {{
    "title": "Animals",
    "author": "Alison Ryan & Julianne Shrum",
    "audio_file": "book_title.mp3"
  }},
  "word_banks": [
     {{
       "page_number": 1,
       "words": [
          {{"word": "example", "file": "wb1_0.mp3"}},
          {{"word": "hats", "file": "wb1_1.mp3"}}
       ]
     }}
  ],
  "story_pages": [
     {{
       "page_number": 1,
       "segments": [
         {{
           "display_text": "\"Here is a dog.\"",
           "speech_script": "Here is a dog.",
           "audio_file": "page1_seg0.mp3"
         }},
         {{
           "display_text": "\"The dog is yellow.\"",
           "speech_script": "The dog is yellow.",
           "audio_file": "page1_seg1.mp3"
         }}
       ],
       "image_file": "page1.png"
     }},
     {{
       "page_number": 2,
       "segments": [
         {{
           "display_text": "\"Here is a cat.\"",
           "speech_script": "The cat is white and brown.",
           "audio_file": "page2_seg0.mp3"
         }},
         {{
           "display_text": "\"The cat is white and brown.\"",
           "speech_script": "The cat is white and brown.",
           "audio_file": "page2_seg1.mp3"
         }}
       ],
       "image_file": "page2.png"
     }}
  ]
}}

Raw Content:
{full_story_text}
"""

print("🤖 Querying Gemini to dynamically handle separate text segments...")
response = ai_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(response_mime_type="application/json"),
)

storybook_manifest = json.loads(response.text)
print("✔️ Text structures analyzed and sequential highlight layout compiled successfully!")

# Extract Story_Author from the new book_metadata structure
Story_Author = storybook_manifest['book_metadata']['author']

print("🖼️ Generating and cropping images for logical story pages...")

def get_physical_page_for_logical(logical_page_num_in_manifest):
    # Now, each logical page directly maps to a physical page.
    return logical_page_num_in_manifest

for story_page_data in storybook_manifest['story_pages']:
    logical_page_num_in_manifest = story_page_data['page_number']
    output_image_filename = story_page_data['image_file']
    physical_page_num = get_physical_page_for_logical(logical_page_num_in_manifest)

    pdf_page = doc[physical_page_num - 1] # Adjust for 0-indexed doc pages

    # No clipping: save the entire physical page as the image for the logical story page
    cropped_pix = pdf_page.get_pixmap(dpi=150)
    cropped_pix.save(os.path.join(images_dir, output_image_filename))

print("✔️ All logical story page images generated without cropping.")

⚙️ Installing processing dependencies (PyMuPDF, edge-tts, google-genai, PyTesseract, Pillow)...
📖 Slicing PDF matrix and collecting raw page text via OCR...
🤖 Querying Gemini to dynamically handle separate text segments...
✔️ Text structures analyzed and sequential highlight layout compiled successfully!
🖼️ Generating and cropping images for logical story pages...
✔️ All logical story page images generated without cropping.


In [ ]:
# @title 🧪 TEST: Check Title page Information
import re

# Extract Story_Author dynamically
# Look for 'by' followed by any characters (including newlines) until 'Animals' or another identifiable section break
author_section_match = re.search(r'by\s+(.*?)\nAnimals', full_story_text, re.DOTALL)
if author_section_match:
    raw_authors_string = author_section_match.group(1).strip()
    # Replace newline + '&' with just '&' to handle multi-line authors, then replace standalone newlines with spaces
    cleaned_authors_string = raw_authors_string.replace('\n&', '&').replace('\n', ' ')

    # Split by '&' to get individual author names
    author_names = [name.strip() for name in cleaned_authors_string.split('&')]
    Story_Author = ' & '.join(author_names)
else:
    Story_Author = "Unknown Author"

print(f"Detected Story Author: {Story_Author}")

Now that the author is dynamically extracted, let's update the web app's layout to place images and text side-by-side.

In [8]:
# ==============================================================================
# CELL 2: VOCAL RENDERING ENGINE
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Loops through the dynamic word banks and narrative steps.
# 2. If it encounters the designated 'is_first_story_page' flag, it renders an
#    isolated voice track exclusively for the title block, then builds the separate
#    body sentence track.
# ==============================================================================

# @title 🎤 Step 2: Render Audio Assets { display-mode: "form" }
Voice_Model = "en-US-GuyNeural"  # @param ["en-US-GuyNeural", "en-US-JennyNeural", "en-GB-SoniaNeural", "en-AU-WilliamNeural"]
Word_Bank_Speed = "-10%"  # @param ["-0%", "-5%", "-10%", "-15%", "-20%"]
Story_Reading_Speed = "-15%"  # @param ["-0%", "-5%", "-10%", "-15%", "-20%", "-25%"]

import edge_tts

async def process_vocal_pipeline():
    book_metadata = storybook_manifest.get("book_metadata", {})
    wb_list = storybook_manifest.get("word_banks", [])
    story_list = storybook_manifest.get("story_pages", [])

    # Render audio for the book title
    if book_metadata and "title" in book_metadata and "audio_file" in book_metadata:
        print("🎙️ Rendering book title audio...")
        title_path = os.path.join(audio_dir, book_metadata['audio_file'])
        title_comm = edge_tts.Communicate(book_metadata['title'], Voice_Model, rate=Story_Reading_Speed)
        await title_comm.save(title_path)

    print(f"🎙️ Rendering {len(wb_list)} vocabulary word sets...")
    for wb_idx, bank in enumerate(wb_list):
        bank_id = wb_idx + 1
        for word_idx, w_item in enumerate(bank.get("words", [])):
            filename = f"wb{bank_id}_{word_idx}.mp3"
            w_item['file'] = filename
            path = os.path.join(audio_dir, filename)
            comm = edge_tts.Communicate(w_item['word'], Voice_Model, rate=Word_Bank_Speed)
            await comm.save(path)

    print(f"🎙️ Rendering narrative pages...")
    for page_idx, page in enumerate(story_list):
        for seg_idx, segment in enumerate(page.get('segments', [])):
            filename = f"page{page['page_number']}_seg{seg_idx}.mp3" # Generate unique filename for each segment
            segment['audio_file'] = filename # Update the manifest with the new filename
            path = os.path.join(audio_dir, filename)
            comm = edge_tts.Communicate(segment['speech_script'], Voice_Model, rate=Story_Reading_Speed)
            await comm.save(path)

    print("✔️ Audio processing execution complete! All structural files rendered.")

await process_vocal_pipeline()

🎙️ Rendering book title audio...
🎙️ Rendering 0 vocabulary word sets...
🎙️ Rendering narrative pages...
✔️ Audio processing execution complete! All structural files rendered.


In [10]:
# ==============================================================================
# CELL 3: APPLICATION COMPILER & DISTRIBUTION PACKAGER
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Dynamically constructs the visual presentation layout.
# 2. Separates the text section on Page 1 into distinct Title and Story components.
# 3. Features an advanced sequential JavaScript audio queue handler that transitions
#    highlights from the Title component down to the Story text component.
# ==============================================================================

# @title 📦 Step 3a: Compile Public Web App & Download Distribution Package
import zipfile
import os
from google.colab import files

wb_list = storybook_manifest.get("word_banks", [])
story_list = storybook_manifest.get("story_pages", [])
book_metadata = storybook_manifest.get("book_metadata", {})

# Build dynamic vocabulary sheets
word_banks_html_stack = ""
for wb_idx, bank in enumerate(wb_list):
    bank_id = wb_idx + 1
    items_html = "".join([f'\n            <div class="word-item" onclick="playAudioFile(this, \'{w["file"]}\')">{w["word"]}</div>' for w in bank.get("words", [])])
    word_banks_html_stack += f"""
    <div class="page-card">
        <div class="page-number">Word Bank {bank_id}</div>
        <p style="margin:0 0 10px 0; font-weight:bold; color:#777;">Vocabulary Review:</p>
        <div class="word-bank">{items_html}</div>
    </div>"""

# Reintroduce the dedicated HTML block for the story title and author as a 'page-card'
title_card_html = f"""
    <div class="page-card" id="title-page-card" style="background-color: var(--primary);">
        <div class="page-number">Title Page</div>
        <div style="text-align: center; padding: 0.625px 0;">
            <h1 style="font-size: 2.8rem; margin-bottom: 8px; color: var(--secondary); display: inline-block;">{book_metadata.get('title', Story_Title)}</h1>
            <h3 style="font-size: 1.2rem; color: #555; margin-top: 5px;">by {book_metadata.get('author', Story_Author)}</h3>
            <p style="font-size: 0.95rem; color: #777; margin-top: 15px;">Interactive Read-Along Experience</p>
        </div>
    </div>"""

# Initialize story_pages_html_stack as empty
story_pages_html_stack = ""

# Build story pages layout columns - starting from the actual first story content page
for page_idx, page in enumerate(story_list):
    segments_html = ""
    for seg_idx, segment in enumerate(page.get('segments', [])):
        segments_html += f"""
                <div class="story-segment" onclick="playAudioFile(this, '{segment['audio_file']}')">
                    {segment['display_text']}
                </div>"""

    title_for_first_story_page = ""
    if page_idx == 0:
        # Add the book title at the top of the first story page
        title_for_first_story_page = f"""
            <div class="story-segment" onclick="playAudioFile(this, '{book_metadata.get('audio_file', 'book_title.mp3')}')" style="display: block; text-align: center; margin-bottom: 20px; font-weight: bold;">
                <h1 style="font-size: 2.5rem; color: var(--secondary); margin-bottom: 5px;">{book_metadata.get('title', Story_Title)}</h1>
            </div>"""

    text_panel_content = f"""
            <div class="story-text-container">
                {title_for_first_story_page}
                {segments_html}
            </div>"""

    story_pages_html_stack += f"""
    <div class="page-card" data-page-number="{page['page_number']}">
        <div class="page-number">Page {page['page_number']}</div>
        <div class="story-layout">
            <div class="illustration-side">
                <img src="storybook_images/{page['image_file']}" alt="Page {page['page_number']}" class="story-image">
            </div>
            <div class="text-side">
                {text_panel_content}
            </div>
        </div>
    </div>"""

complete_html_payload = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{Story_Title}</title>
    <style>
        :root {{ --primary: #FFD700; --secondary: #333333; --background: #eef2f5; --card-bg: #ffffff; --text-color: #2c3e50; --interactive-hover: #eaf5ff; --interactive-active: #c6e4ff; --button-bg: #4CAF50; --button-color: white; }}
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: var(--background); color: var(--text-color); margin: 0; padding: 20px; display: flex; flex-direction: column; align-items: center; }}
        .container {{ max-width: 850px; width: 100%; }}
        header {{ display: none; }} /* Hide global header now that title card is back */
        .audio-hint {{ display: flex; align-items: center; justify-content: center; gap: 8px; margin-bottom: 25px; font-size: 0.95rem; color: #555; font-weight: 500; }}
        .audio-hint svg {{ width: 20px; height: 20px; fill: currentColor; }}
        .page-card {{ background-color: var(--card-bg); border-radius: 16px; box-shadow: 0 6px 20px rgba(0,0,0,0.05); padding: 25px; margin-bottom: 30px; position: relative; box-sizing: border-box; }}
        .story-layout {{ display: flex; gap: 25px; align-items: center; margin-top: 10px; }}
        .illustration-side {{ flex: 0 0 55%; max-width: 55%; }} /* Reverted max-width to 55% */
        .story-image {{ width: 100%; height: auto; display: block; border-radius: 12px; border: 3px solid var(--secondary); box-shadow: 0 4px 10px rgba(0,0,0,0.08); background-color: #fafafa; }}
        .text-side {{ flex: 1; display: flex; flex-direction: column; }}
        .page-number {{ position: absolute; top: 15px; right: 20px; font-size: 0.8rem; font-weight: bold; color: #bbb; text-transform: uppercase; letter-spacing: 1px; }}
        .word-bank {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(110px, 1fr)); gap: 12px; margin-top: 10px; }}
        .word-item:hover {{ background-color: #f0fdf4; border-color: #4ade80; transform: scale(1.03); }}
        .speaking {{ background-color: var(--interactive-active) !important; color: #004499 !important; border-color: #0066cc !important; }}
        .story-segment {{ padding: 8px 12px; margin-bottom: 8px; border-radius: 8px; cursor: pointer; transition: all 0.2s ease; border: 1px solid transparent; font-size: 1.5rem; }}
        .story-segment:hover {{ background-color: var(--interactive-hover); border-color: var(--interactive-active); }}
        .play-button {{ background-color: var(--button-bg); color: var(--button-color); border: none; padding: 10px 20px; border-radius: 8px; cursor: pointer; font-size: 1rem; margin-top: 20px; transition: background-color 0.2s ease; }}
        .play-button:hover {{ background-color: #45a049; }}
        @media(max-width: 768px) {{ .story-layout {{ flex-direction: column; }} .illustration-side {{ flex: 0 0 100%; max-width: 100%; }} }}
    </style>
</head>
<body>
<div class="container">
    {title_card_html}
    <div class="audio-hint">
        <svg viewBox="0 0 24 24"><path d="M3 9v6h4l5 5V4L7 9H3zm13.5 3c0-1.77-1.02-3.25-2.5-4.01v8.03c1.48-.76 2.5-2.24 2.5-4.02zM14 3.23v2.06c2.89.86 5 3.54 5 6.71s-2.11 5.85-5 6.71v2.06c4.01-.91 7-4.49 7-8.77s-2.99-7.86-7-8.77z"/></svg>
        <span>Click elements to play professional voice track!</span>
    </div>
    {word_banks_html_stack}
    {story_pages_html_stack}
</div>
<script>
    let currentAudio = null; let currentElement = null;

    function resetActiveStates() {{
        if (currentAudio) {{ currentAudio.pause(); currentAudio.currentTime = 0; }}
        if (currentElement) {{ currentElement.classList.remove('speaking'); }}
    }}

    function playAudioFile(element, filename, onendedCallback = null) {{
        resetActiveStates();
        currentElement = element; currentElement.classList.add('speaking');

        let src = filename.startsWith('storybook_audios/') ? filename : 'storybook_audios/' + filename;
        currentAudio = new Audio(src);
        currentAudio.play().catch(error => {{ console.log("Playback issue:", error); currentElement.classList.remove('speaking'); }});
        currentAudio.onended = function() {{
            if (currentElement) {{ currentElement.classList.remove('speaking'); }}
            if (onendedCallback) {{ onendedCallback(); }}
        }};
    }}
</script>
</body>
</html>"""

with open(os.path.join(project_root, "index.html"), "w", encoding="utf-8") as f:
    f.write(complete_html_payload)

print("📦 Packaging pipeline zip module distribution payload...")
zip_filename = f"{Folder_Safe_Name}_production_package.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, file_list in os.walk(project_root):
        for file in file_list:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(project_root))
            zipf.write(file_path, arcname)

files.download(zip_filename)
print("🚀 Complete bundle dispatched directly to downloads archive!")

📦 Packaging pipeline zip module distribution payload...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 Complete bundle dispatched directly to downloads archive!


In [9]:
# ==============================================================================
# CELL 3b: APPLICATION COMPILER & DISTRIBUTION PACKAGER (with SAS Token)
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Dynamically constructs the visual presentation layout.
# 2. Separates the text section on Page 1 into distinct Title and Story components.
# 3. Features an advanced sequential JavaScript audio queue handler that transitions
#    highlights from the Title component down to the Story text component.
# 4. Appends a SAS token to all audio and image asset URLs dynamically from the URL.
# ==============================================================================

# @title 📦 Step 3b: Compile Private Web App & Download Distribution Package
import zipfile
import os
from google.colab import files

wb_list = storybook_manifest.get("word_banks", [])
story_list = storybook_manifest.get("story_pages", [])
book_metadata = storybook_manifest.get("book_metadata", {})

# Build dynamic vocabulary sheets
word_banks_html_stack = ""
for wb_idx, bank in enumerate(wb_list):
    bank_id = wb_idx + 1
    items_html = "".join([
f"            <div class=\"word-item\" onclick=\"playAudioFile(this, '{w["file"]}')\">{w["word"]}</div>" for w in bank.get("words", [])])
    word_banks_html_stack += f"""
    <div class="page-card">
        <div class="page-number">Word Bank {bank_id}</div>
        <p style="margin:0 0 10px 0; font-weight:bold; color:#777;">Vocabulary Review:</p>
        <div class="word-bank">{items_html}</div>
    </div>"""

# Reintroduce the dedicated HTML block for the story title and author as a 'page-card'
title_card_html = f"""
    <div class="page-card" id="title-page-card" style="background-color: var(--primary);">
        <div class="page-number">Title Page</div>
        <div style="text-align: center; padding: 0.625px 0;">
            <h1 style="font-size: 2.8rem; margin-bottom: 8px; color: var(--secondary); display: inline-block;">{book_metadata.get('title', Story_Title)}</h1>
            <h3 style="font-size: 1.2rem; color: #555; margin-top: 5px;">by {book_metadata.get('author', Story_Author)}</h3>
            <p style="font-size: 0.95rem; color: #777; margin-top: 15px;">Interactive Read-Along Experience</p>
        </div>
    </div>"""

# Initialize story_pages_html_stack as empty
story_pages_html_stack = ""

# Build story pages layout columns - starting from the actual first story content page
for page_idx, page in enumerate(story_list):
    segments_html = ""
    for seg_idx, segment in enumerate(page.get('segments', [])):
        segments_html += f"""
                <div class="story-segment" onclick=\"playAudioFile(this, '{segment['audio_file']}')\">
                    {segment['display_text']}
                </div>"""

    title_for_first_story_page = ""
    if page_idx == 0:
        # Add the book title at the top of the first story page
        title_for_first_story_page = f"""
            <div class="story-segment" onclick=\"playAudioFile(this, '{book_metadata.get('audio_file', 'book_title.mp3')}')\" style="display: block; text-align: center; margin-bottom: 20px; font-weight: bold;">
                <h1 style="font-size: 2.5rem; color: var(--secondary); margin-bottom: 5px;">{book_metadata.get('title', Story_Title)}</h1>
            </div>"""

    text_panel_content = f"""
            <div class="story-text-container">
                {title_for_first_story_page}
                {segments_html}
            </div>"""

    story_pages_html_stack += f"""
    <div class="page-card" data-page-number="{page['page_number']}">
        <div class="page-number">Page {page['page_number']}</div>
        <div class="story-layout">
            <div class="illustration-side">
                <img src="storybook_images/{page['image_file']}" alt="Page {page['page_number']}" class="story-image">
            </div>
            <div class="text-side">
                {text_panel_content}
            </div>
        </div>
    </div>"""

complete_html_payload = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{Story_Title}</title>
    <style>
        :root {{ --primary: #FFD700; --secondary: #333333; --background: #eef2f5; --card-bg: #ffffff; --text-color: #2c3e50; --interactive-hover: #eaf5ff; --interactive-active: #c6e4ff; --button-bg: #4CAF50; --button-color: white; }}
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: var(--background); color: var(--text-color); margin: 0; padding: 20px; display: flex; flex-direction: column; align-items: center; }}
        .container {{ max-width: 850px; width: 100%; }}
        header {{ display: none; }} /* Hide global header now that title card is back */
        .audio-hint {{ display: flex; align-items: center; justify-content: center; gap: 8px; margin-bottom: 25px; font-size: 0.95rem; color: #555; font-weight: 500; }}
        .audio-hint svg {{ width: 20px; height: 20px; fill: currentColor; }}
        .page-card {{ background-color: var(--card-bg); border-radius: 16px; box-shadow: 0 6px 20px rgba(0,0,0,0.05); padding: 25px; margin-bottom: 30px; position: relative; box-sizing: border-box; }}
        .story-layout {{ display: flex; gap: 25px; align-items: center; margin-top: 10px; }}
        .illustration-side {{ flex: 0 0 55%; max-width: 55%; }} /* Reverted max-width to 55% */
        .story-image {{ width: 100%; height: auto; display: block; border-radius: 12px; border: 3px solid var(--secondary); box-shadow: 0 4px 10px rgba(0,0,0,0.08); background-color: #fafafa; }}
        .text-side {{ flex: 1; display: flex; flex-direction: column; }}
        .page-number {{ position: absolute; top: 15px; right: 20px; font-size: 0.8rem; font-weight: bold; color: #bbb; text-transform: uppercase; letter-spacing: 1px; }}
        .word-bank {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(110px, 1fr)); gap: 12px; margin-top: 10px; }}
        .word-item:hover {{ background-color: #f0fdf4; border-color: #4ade80; transform: scale(1.03); }}
        .speaking {{ background-color: var(--interactive-active) !important; color: #004499 !important; border-color: #0066cc !important; }}
        .story-segment {{ padding: 8px 12px; margin-bottom: 8px; border-radius: 8px; cursor: pointer; transition: all 0.2s ease; border: 1px solid transparent; font-size: 1.5rem; }}
        .story-segment:hover {{ background-color: var(--interactive-hover); border-color: var(--interactive-active); }}
        .play-button {{ background-color: var(--button-bg); color: var(--button-color); border: none; padding: 10px 20px; border-radius: 8px; cursor: pointer; font-size: 1rem; margin-top: 20px; transition: background-color 0.2s ease; }}
        .play-button:hover {{ background-color: #45a049; }}
        @media(max-width: 768px) {{ .story-layout {{ flex-direction: column; }} .illustration-side {{ flex: 0 0 100%; max-width: 100%; }} }}
    </style>
</head>
<body>
<div class="container">
    {title_card_html}
    <div class="audio-hint">
        <svg viewBox="0 0 24 24"><path d="M3 9v6h4l5 5V4L7 9H3zm13.5 3c0-1.77-1.02-3.25-2.5-4.01v8.03c1.48-.76 2.5-2.24 2.5-4.02zM14 3.23v2.06c2.89.86 5 3.54 5 6.71s-2.11 5.85-5 6.71v2.06c4.01-.91 7-4.49 7-8.77s-2.99-7.86-7-8.77z"/></svg>
        <span>Click elements to play professional voice track!</span>
    </div>
    {word_banks_html_stack}
    {story_pages_html_stack}
</div>
<script>
    let currentAudio = null; let currentElement = null;

    function resetActiveStates() {{
        if (currentAudio) {{ currentAudio.pause(); currentAudio.currentTime = 0; }}
        if (currentElement) {{ currentElement.classList.remove('speaking'); }}
    }}

    function playAudioFile(element, filename) {{
        resetActiveStates();
        currentElement = element;
        currentElement.classList.add('speaking');

        // Let the overridden Audio() constructor handle SAS + path rewriting
        currentAudio = new Audio(filename);

        currentAudio.play().catch(error => {{
            console.log("Playback issue:", error);
            if (currentElement) currentElement.classList.remove('speaking');
        }});

        currentAudio.onended = function () {{
            if (currentElement) {{
                currentElement.classList.remove('speaking');
            }}
        }};
    }}

    (function () {{
        const sas = window.location.search;
        if (!sas) return;

        const BASE = window.location.origin + "/sarareadingprivate/" + "{Folder_Safe_Name}" + "/";

        function resolveAsset(src) {{
            if (!src) return src;
            if (src.startsWith("http")) return src;

            // remove leading slashes
            src = src.replace(/^\\/+/, "");

            let resolvedSrc = src;
            if (!src.startsWith("storybook_images/") && !src.startsWith("storybook_audios/")) {{
                if (src.endsWith(".mp3")) {{
                    resolvedSrc = "storybook_audios/" + src;
                }} else {{
                    resolvedSrc = "storybook_images/" + src; // Default to images
                }}
            }}
            return BASE + resolvedSrc + sas;
        }}

        // FIX IMAGES
        document.querySelectorAll("img").forEach(img => {{
            const src = img.getAttribute("src");
            img.src = resolveAsset(src);
        }});

        // FIX AUDIO USAGE
        const originalAudio = window.Audio;

        window.Audio = function (src) {{
            return new originalAudio(resolveAsset(src));
        }};

    }})();
</script>
</body>
</html>"""

with open(os.path.join(project_root, "index_sas.html"), "w", encoding="utf-8") as f:
    f.write(complete_html_payload)

print("📦 Packaging pipeline zip module distribution payload (with SAS token)...")
zip_filename = f"{Folder_Safe_Name}_production_package_sas.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Write the SAS-enabled index.html
    zipf.write(os.path.join(project_root, "index_sas.html"), os.path.join(project_root, "index.html"))

    # Add other assets (audio and images)
    for folder in [audio_dir, images_dir]:
        for root, _, file_list in os.walk(folder):
            for file in file_list:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, os.path.dirname(project_root)) # Keep original folder structure
                zipf.write(file_path, arcname)

files.download(zip_filename)
print("🚀 Complete bundle with SAS token dispatched directly to downloads archive!")

📦 Packaging pipeline zip module distribution payload (with SAS token)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 Complete bundle with SAS token dispatched directly to downloads archive!


In [ ]:
# ==============================================================================
# CELL 4: WORKSPACE CLEANUP ENGINE
# ------------------------------------------------------------------------------
# WHAT THIS CELL DOES:
# 1. Provides a graphical form field for you to specify any target directory.
# 2. Checks if that specific folder exists in your current Colab environment.
# 3. Recursively deletes all files, images, audios, and subfolders inside it
#    to cleanly reset your workspace.
# ==============================================================================

# @title 🧹 Step 4: Clear Specified Project Directory { display-mode: "form" }
target_directory = "30_MrBeeAndTed"  # @param {type:"string"}

import shutil
import os

# Clean up any trailing slashes user might type by accident
target_directory = target_directory.strip().rstrip('/')

if target_directory and os.path.exists(target_directory):
    print(f"⚠️ Initializing cleanup sequence for directory: '{target_directory}'...")

    try:
        # Recursively deletes the entire directory tree
        shutil.rmtree(target_directory)
        print(f"🗑️ Clean sweep complete! '{target_directory}' and all of its contents have been deleted.")
    except Exception as e:
        print(f"❌ An error occurred while trying to delete the folder: {e}")

elif not target_directory:
    print("❌ Error: Please specify a valid folder name in the target_directory box.")
else:
    print(f"ℹ️ The folder '{target_directory}' does not exist or has already been cleared.")